# 🔥 EmberNote
### Aplikasi Pesan Rahasia Sekali Baca (Burn-After-Reading) Berbasis AES-256-GCM

**Tema:** komunikasi rahasia sementara, pesan hanya bisa dibuka satu kali lalu hangus.

**Deskripsi.** EmberNote mengenkripsi pesan dengan **satu algoritma saja, AES-256-GCM**. Kunci 256-bit dibuat acak, nonce unik dibuat di setiap enkripsi, dan metadata (ID pesan + waktu kedaluwarsa) diikat ke ciphertext lewat **AAD**. Server hanya menyimpan ciphertext. Saat dibuka, tag GCM diverifikasi: kunci salah atau data/metadata yang diubah langsung ditolak. Setelah berhasil dibaca, pesan dihapus dari server.

**Mengapa cukup satu algoritma?** AES-GCM sudah memberi *kerahasiaan* **dan** *integritas* sekaligus, sehingga tidak perlu cipher klasik, XOR, RC4, ataupun hash/MAC tambahan. Bagian 3 membuktikannya lewat sembilan pengujian.

## 0. Persiapan dan Struktur Proyek
```
EmberNote/
├── .streamlit/config.toml
├── images/logo.png
├── modules/  (aes_gcm.py, storage.py, selftest.py)
├── app.py   tema.py   requirements.txt   .gitignore
└── laporan.ipynb
```
Sel-sel di bawah menulis file-file tersebut ke folder kerja notebook. (`images/logo.png` ada di dalam zip; aplikasi tetap jalan tanpanya.)

In [ ]:
!pip install -q cryptography streamlit

In [ ]:
import os
for d in ["modules", ".streamlit", "images"]:
    os.makedirs(d, exist_ok=True)
with open("modules/__init__.py", "w", encoding="utf-8") as f:
    f.write('"""Modul kriptografi EmberNote: hanya AES-256-GCM."""\n')

## 1. Inti Kriptografi (`modules/aes_gcm.py`)
Hanya memakai `AESGCM` dari library `cryptography`. Tag autentikasi 16 byte menempel di akhir ciphertext.

In [ ]:
%%writefile modules/aes_gcm.py
"""EmberNote - inti kriptografi.

Satu-satunya algoritma yang dipakai: AES-256-GCM (authenticated encryption).
  - kerahasiaan  : isi pesan dienkripsi (AES-256 mode counter)
  - integritas   : authentication tag 128-bit (GHASH) menolak data yang diubah
  - keaslian metadata : AAD (ID pesan + waktu kedaluwarsa) ikut diautentikasi
Kunci dibuat acak oleh CSPRNG sistem operasi, bukan diturunkan dari password,
sehingga tidak ada algoritma kedua (KDF/hash) di dalam aplikasi.
"""
import base64
import json
import os
import secrets

from cryptography.exceptions import InvalidTag
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

KEY_BYTES = 32     # 256 bit
NONCE_BYTES = 12   # 96 bit, ukuran standar GCM


class DecryptionError(Exception):
    """Dekripsi ditolak: kunci salah, ciphertext diubah, atau metadata (AAD) diubah."""


def generate_key() -> bytes:
    """Kunci AES-256 acak (32 byte)."""
    return AESGCM.generate_key(bit_length=256)


def generate_id() -> str:
    """ID pesan acak (bukan rahasia kriptografis, hanya penanda di server)."""
    return secrets.token_urlsafe(12)


def b64_encode(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode().rstrip("=")


def b64_decode(text: str) -> bytes:
    text = text.strip()
    return base64.urlsafe_b64decode(text + "=" * (-len(text) % 4))


def parse_key(text: str) -> bytes:
    """Ubah teks kunci (Base64 URL-safe) menjadi 32 byte, atau tolak."""
    try:
        key = b64_decode(text)
    except Exception as exc:
        raise DecryptionError("Format kunci tidak valid.") from exc
    if len(key) != KEY_BYTES:
        raise DecryptionError("Format kunci tidak valid.")
    return key


def build_aad(msg_id: str, expires_at: int) -> bytes:
    """Associated Data: metadata yang tidak dienkripsi tetapi ikut diautentikasi."""
    return json.dumps(
        {"id": msg_id, "exp": int(expires_at)}, sort_keys=True, separators=(",", ":")
    ).encode()


def encrypt(plaintext: str, key: bytes, aad: bytes):
    """Return (nonce, ciphertext). Tag 16 byte menempel di akhir ciphertext."""
    if len(key) != KEY_BYTES:
        raise ValueError("Kunci harus 32 byte (AES-256).")
    nonce = os.urandom(NONCE_BYTES)  # nonce baru di setiap enkripsi
    ciphertext = AESGCM(key).encrypt(nonce, plaintext.encode("utf-8"), aad)
    return nonce, ciphertext


def decrypt(nonce: bytes, ciphertext: bytes, key: bytes, aad: bytes) -> str:
    """Dekripsi + verifikasi tag. Jika gagal, tidak ada output sebagian sama sekali."""
    try:
        return AESGCM(key).decrypt(nonce, ciphertext, aad).decode("utf-8")
    except (InvalidTag, ValueError) as exc:
        raise DecryptionError(
            "Kunci salah, atau data/metadata pesan telah diubah (tag GCM tidak cocok)."
        ) from exc

In [ ]:
from modules import aes_gcm as cc

key = cc.generate_key()
msg_id = cc.generate_id()
aad = cc.build_aad(msg_id, 1893456000)

nonce, ct = cc.encrypt("Halo, ini pesan rahasia!", key, aad)
print("Ciphertext+tag (hex):", ct.hex())
print("Dekripsi            :", cc.decrypt(nonce, ct, key, aad))

## 2. Penyimpanan Server (`modules/storage.py`)
Tabel SQLite berisi `id, nonce, ciphertext, expires_at, created_at`. **Tidak ada kolom untuk kunci atau plaintext.** Penghapusan memakai satu perintah `DELETE` atomik, jadi hanya satu pembaca yang berhasil.

In [ ]:
%%writefile modules/storage.py
"""Penyimpanan server (SQLite). Server HANYA menyimpan ciphertext, tidak pernah kunci/plaintext."""
import sqlite3
import time
from contextlib import closing

DB_PATH = "embernote.db"


def _connect(db_path: str) -> sqlite3.Connection:
    conn = sqlite3.connect(db_path, timeout=10, isolation_level=None)  # autocommit
    conn.execute(
        """CREATE TABLE IF NOT EXISTS messages (
               id TEXT PRIMARY KEY,
               nonce BLOB NOT NULL,
               ciphertext BLOB NOT NULL,
               expires_at INTEGER NOT NULL,
               created_at INTEGER NOT NULL
           )"""
    )
    conn.execute("DELETE FROM messages WHERE expires_at <= ?", (int(time.time()),))
    return conn


def save_message(msg_id, nonce, ciphertext, expires_at, db_path=DB_PATH) -> None:
    with closing(_connect(db_path)) as conn:
        conn.execute(
            "INSERT INTO messages VALUES (?, ?, ?, ?, ?)",
            (msg_id, nonce, ciphertext, int(expires_at), int(time.time())),
        )


def fetch_message(msg_id, db_path=DB_PATH):
    """Ambil pesan (tanpa menghapus). None jika tidak ada / sudah kedaluwarsa."""
    with closing(_connect(db_path)) as conn:
        row = conn.execute(
            "SELECT id, nonce, ciphertext, expires_at, created_at FROM messages WHERE id = ?",
            (msg_id,),
        ).fetchone()
    if row is None:
        return None
    return {
        "id": row[0], "nonce": row[1], "ciphertext": row[2],
        "expires_at": row[3], "created_at": row[4],
    }


def consume_message(msg_id, db_path=DB_PATH) -> bool:
    """Hapus pesan secara atomik. True hanya untuk SATU pemanggil yang berhasil menghapus."""
    with closing(_connect(db_path)) as conn:
        return conn.execute("DELETE FROM messages WHERE id = ?", (msg_id,)).rowcount == 1

## 3. Pengujian Otomatis (`modules/selftest.py`): bukti bahwa hasilnya "habis"
Sembilan pengujian menunjukkan bahwa satu algoritma sudah tuntas: round-trip, kunci salah, ciphertext/tag/metadata diubah, nonce unik, plaintext tidak bocor, sekali baca, dan kedaluwarsa.

In [ ]:
%%writefile modules/selftest.py
"""Pengujian otomatis sebagai BUKTI bahwa satu algoritma (AES-256-GCM) sudah cukup."""
import os
import sqlite3
import tempfile
import time

from . import aes_gcm as cc
from . import storage


def _rejected(func) -> bool:
    try:
        func()
    except cc.DecryptionError:
        return True
    return False


def run_all():
    results = []

    def check(name, ok, detail):
        results.append({"Pengujian": name, "Hasil": "LULUS" if ok else "GAGAL", "Keterangan": detail})

    key = cc.generate_key()
    msg_id = cc.generate_id()
    exp = int(time.time()) + 3600
    aad = cc.build_aad(msg_id, exp)
    plain = "Pesan rahasia uji EmberNote"
    nonce, ct = cc.encrypt(plain, key, aad)

    check("1. Round-trip", cc.decrypt(nonce, ct, key, aad) == plain,
          "Enkripsi lalu dekripsi menghasilkan teks yang sama persis.")

    check("2. Kunci salah ditolak", _rejected(lambda: cc.decrypt(nonce, ct, cc.generate_key(), aad)),
          "Tidak ada output sebagian, dekripsi gagal total.")

    bad = bytearray(ct); bad[0] ^= 0x01
    check("3. Ciphertext diubah 1 bit", _rejected(lambda: cc.decrypt(nonce, bytes(bad), key, aad)),
          "Tag GCM mendeteksi perubahan, tanpa perlu hash/MAC tambahan.")

    bad = bytearray(ct); bad[-1] ^= 0x01
    check("4. Tag diubah", _rejected(lambda: cc.decrypt(nonce, bytes(bad), key, aad)),
          "Authentication tag yang dipalsukan ditolak.")

    check("5. Metadata (AAD) diubah", _rejected(
        lambda: cc.decrypt(nonce, ct, key, cc.build_aad(msg_id, exp + 86400))),
        "Memperpanjang waktu kedaluwarsa pesan membuat dekripsi gagal.")

    n2, ct2 = cc.encrypt(plain, key, aad)
    check("6. Nonce unik", nonce != n2 and ct != ct2,
          "Plaintext sama dienkripsi dua kali menghasilkan ciphertext berbeda.")

    check("7. Plaintext tidak bocor", plain.encode() not in ct,
          "Teks asli tidak muncul di dalam ciphertext yang disimpan.")

    with tempfile.TemporaryDirectory() as tmp:
        db = os.path.join(tmp, "test.db")
        storage.save_message(msg_id, nonce, ct, exp, db)
        first = storage.consume_message(msg_id, db)
        second = storage.consume_message(msg_id, db)
        check("8. Sekali baca", first and not second and storage.fetch_message(msg_id, db) is None,
              "Pembaca pertama berhasil menghapus, pembaca kedua tidak mendapat apa-apa.")

        storage.save_message("kadaluwarsa", nonce, ct, int(time.time()) + 1, db)
        sqlite3.connect(db).execute("UPDATE messages SET expires_at = ?", (int(time.time()) - 1,)).connection.commit()
        check("9. Kedaluwarsa", storage.fetch_message("kadaluwarsa", db) is None,
              "Pesan yang melewati masa berlaku otomatis terhapus.")

    return results


# Jalankan: python -m modules.selftest
if __name__ == "__main__":
    for r in run_all():
        print(f"{r['Hasil']:6} {r['Pengujian']:32} {r['Keterangan']}")

In [ ]:
from modules import selftest

hasil = selftest.run_all()
for r in hasil:
    print(f"{r['Hasil']:6} {r['Pengujian']:30} {r['Keterangan']}")
print(f"\n{sum(r['Hasil']=='LULUS' for r in hasil)} dari {len(hasil)} pengujian lulus")

## 4. Demo Alur Lengkap (tanpa Streamlit)
Menirukan alur pengirim dan penerima, termasuk percobaan serangan.

In [ ]:
import os, time, sqlite3
from modules import aes_gcm as cc, storage

DB = "demo.db"
if os.path.exists(DB): os.remove(DB)

# --- PENGIRIM ---
pesan = "Kode brankas: 4821-XK"
key = cc.generate_key()
msg_id = cc.generate_id()
exp = int(time.time()) + 3600
nonce, ct = cc.encrypt(pesan, key, cc.build_aad(msg_id, exp))
storage.save_message(msg_id, nonce, ct, exp, DB)
print("ID      :", msg_id)
print("Kunci   :", cc.b64_encode(key), "(dikirim lewat jalur lain)")

# --- Isi server: hanya data acak ---
row = storage.fetch_message(msg_id, DB)
print("\nDi server -> nonce:", row["nonce"].hex())
print("Di server -> ciphertext+tag:", row["ciphertext"].hex())
print("Teks asli ada di server?", pesan.encode() in row["ciphertext"])

# --- PENYERANG: kunci salah ---
try:
    cc.decrypt(row["nonce"], row["ciphertext"], cc.generate_key(), cc.build_aad(row["id"], row["expires_at"]))
except cc.DecryptionError as e:
    print("\n[Serangan 1] Kunci salah  ->", e)

# --- PENYERANG: perpanjang masa berlaku di database ---
sqlite3.connect(DB).execute("UPDATE messages SET expires_at = expires_at + 86400").connection.commit()
row2 = storage.fetch_message(msg_id, DB)
try:
    cc.decrypt(row2["nonce"], row2["ciphertext"], key, cc.build_aad(row2["id"], row2["expires_at"]))
except cc.DecryptionError as e:
    print("[Serangan 2] Metadata diubah ->", e)

# --- PENERIMA: pesan asli (database dipulihkan) ---
sqlite3.connect(DB).execute("UPDATE messages SET expires_at = ?", (exp,)).connection.commit()
row3 = storage.fetch_message(msg_id, DB)
isi = cc.decrypt(row3["nonce"], row3["ciphertext"], cc.parse_key(cc.b64_encode(key)), cc.build_aad(row3["id"], row3["expires_at"]))
if storage.consume_message(msg_id, DB):
    print("\n[Penerima] Pesan terbuka:", isi)
print("[Pembaca kedua] consume lagi ->", storage.consume_message(msg_id, DB), "(pesan sudah hangus)")

## 5. Antarmuka Streamlit
Empat halaman di `app.py`: **Buat Pesan**, **Baca Pesan**, **Bukti Keamanan** (menjalankan pengujian langsung), dan **Tentang**. `tema.py` berisi CSS dan komponen header, `.streamlit/config.toml` mengatur warna tema.

Catatan desain: Streamlit tidak dapat membaca bagian `#fragment` dari URL, jadi **ID** dan **kunci** dibagikan terpisah. Ini juga lebih aman karena satu saluran yang bocor tidak cukup untuk membuka pesan.

In [ ]:
%%writefile tema.py
"""Tema visual EmberNote (CSS tambahan di atas .streamlit/config.toml)."""
import streamlit as st

CSS = """
<style>
.block-container { padding-top: 2rem; max-width: 820px; }
.en-hero { padding: 1.3rem 1.6rem; border-radius: 16px; margin-bottom: 1.2rem;
  background: linear-gradient(135deg, #FF6B35 0%, #C2410C 55%, #7C2D12 100%); color: #fff; }
.en-hero h1 { margin: 0; font-size: 2rem; color: #fff; padding: 0; }
.en-hero p { margin: .3rem 0 0; opacity: .92; }
.en-badge { display: inline-block; padding: .15rem .6rem; margin: .6rem .35rem 0 0; border-radius: 999px;
  font-size: .78rem; background: rgba(255,255,255,.18); border: 1px solid rgba(255,255,255,.35); }
div[data-testid="stForm"] { border-radius: 14px; border: 1px solid rgba(255,107,53,.35); }
.stButton > button, div[data-testid="stFormSubmitButton"] > button { border-radius: 10px; font-weight: 600; }
</style>
"""


def apply_theme() -> None:
    st.markdown(CSS, unsafe_allow_html=True)


def hero(title: str, subtitle: str, badges=()) -> None:
    chips = "".join(f'<span class="en-badge">{b}</span>' for b in badges)
    st.markdown(
        f'<div class="en-hero"><h1>{title}</h1><p>{subtitle}</p>{chips}</div>',
        unsafe_allow_html=True,
    )

In [ ]:
%%writefile .streamlit/config.toml
[theme]
base = "dark"
primaryColor = "#FF6B35"
backgroundColor = "#14110F"
secondaryBackgroundColor = "#221B17"
textColor = "#F5EDE6"
font = "sans serif"

[browser]
gatherUsageStats = false

In [ ]:
%%writefile app.py
"""EmberNote - Pesan rahasia sekali baca berbasis AES-256-GCM (Streamlit)."""
import time
from datetime import datetime, timedelta, timezone

from pathlib import Path

import streamlit as st

import tema
from modules import aes_gcm as cc
from modules import selftest, storage

st.set_page_config(page_title="EmberNote", page_icon="🔥", layout="centered")
tema.apply_theme()

LOGO = Path(__file__).parent / "images" / "logo.png"

MAX_CHARS = 5000
DURATIONS = {"5 menit": 300, "1 jam": 3600, "24 jam": 86400, "7 hari": 604800}
PAGES = ["🔥 Buat Pesan", "📖 Baca Pesan", "🛡️ Bukti Keamanan", "ℹ️ Tentang"]
WIB = timezone(timedelta(hours=7))


def fmt_time(ts: int) -> str:
    return datetime.fromtimestamp(ts, WIB).strftime("%d %b %Y %H:%M:%S WIB")


# ---------------------------------------------------------------- Buat pesan
def page_create():
    st.header("🔥 Buat Pesan Rahasia Sekali Baca")
    st.caption("Pesan dienkripsi dengan AES-256-GCM, dibuka sekali, lalu hangus.")

    with st.form("form_buat", clear_on_submit=True):
        text = st.text_area("Tulis pesan rahasiamu", max_chars=MAX_CHARS, height=180)
        label = st.selectbox("Masa berlaku pesan", list(DURATIONS))
        submitted = st.form_submit_button("🔒 Enkripsi & buat pesan", type="primary")

    if submitted:
        if not text.strip():
            st.warning("Pesan tidak boleh kosong.")
        else:
            key = cc.generate_key()
            msg_id = cc.generate_id()
            expires_at = int(time.time()) + DURATIONS[label]
            aad = cc.build_aad(msg_id, expires_at)
            nonce, ciphertext = cc.encrypt(text, key, aad)
            storage.save_message(msg_id, nonce, ciphertext, expires_at)
            st.session_state.created = {
                "id": msg_id, "key": cc.b64_encode(key), "exp": expires_at,
            }

    created = st.session_state.get("created")
    if not created:
        return

    st.success("Pesan terenkripsi dan tersimpan. Kirim ID dan kunci lewat jalur yang BERBEDA.")
    st.markdown("**ID Pesan** (boleh dikirim lewat chat biasa)")
    st.code(created["id"], language=None)
    st.markdown("**Kunci Rahasia** (kirim lewat jalur lain; hanya ditampilkan sekarang)")
    st.code(created["key"], language=None)
    st.info(
        f"Berlaku sampai {fmt_time(created['exp'])}. Tautan langsung ke halaman baca: "
        f"tambahkan `?id={created['id']}` di akhir alamat aplikasi."
    )

    with st.expander("🔍 Apa yang sebenarnya tersimpan di server?"):
        row = storage.fetch_message(created["id"])
        if row:
            st.json({
                "id": row["id"],
                "nonce (hex)": row["nonce"].hex(),
                "ciphertext + tag (hex)": row["ciphertext"].hex(),
                "expires_at": row["expires_at"],
            })
            st.caption("Server tidak menyimpan kunci maupun teks asli, hanya data acak di atas.")

    if st.button("🧹 Bersihkan tampilan (hapus kunci dari layar)"):
        st.session_state.pop("created", None)
        st.rerun()


# ---------------------------------------------------------------- Baca pesan
def page_read():
    st.header("📖 Baca Pesan Rahasia")

    revealed = st.session_state.get("revealed")
    if revealed is not None:
        st.success("Pesan berhasil dibuka dan SUDAH DIHAPUS dari server. Simpan jika perlu.")
        st.text_area("Isi pesan", value=revealed, height=200)
        if st.button("Tutup pesan"):
            st.session_state.revealed = None
            st.rerun()
        return

    with st.form("form_baca"):
        msg_id = st.text_input("ID Pesan", value=st.query_params.get("id", ""))
        key_text = st.text_input("Kunci Rahasia", type="password")
        submitted = st.form_submit_button("🔓 Buka pesan (hanya bisa sekali)", type="primary")

    if not submitted:
        return

    msg_id = msg_id.strip()
    row = storage.fetch_message(msg_id)
    if row is None:
        st.error("Pesan tidak ditemukan, sudah dibaca, atau sudah kedaluwarsa.")
        return
    try:
        key = cc.parse_key(key_text)
        aad = cc.build_aad(row["id"], row["expires_at"])
        plaintext = cc.decrypt(row["nonce"], row["ciphertext"], key, aad)
    except cc.DecryptionError as exc:
        st.error(f"Gagal membuka pesan: {exc}")
        return

    # Hapus dulu secara atomik; hanya satu pembaca yang boleh melihat isi pesan.
    if storage.consume_message(msg_id):
        st.session_state.revealed = plaintext
        st.rerun()
    else:
        st.error("Pesan ini baru saja dibuka oleh pihak lain.")


# ----------------------------------------------------------- Bukti keamanan
def page_proof():
    st.header("🛡️ Bukti Keamanan")
    st.write(
        "Halaman ini menjalankan pengujian langsung untuk menunjukkan bahwa **satu algoritma, "
        "AES-256-GCM, sudah mencukupi**: menjaga kerahasiaan, mendeteksi data yang diubah, "
        "dan melindungi metadata, tanpa hash atau cipher tambahan."
    )
    if st.button("▶️ Jalankan semua pengujian", type="primary"):
        results = selftest.run_all()
        passed = sum(r["Hasil"] == "LULUS" for r in results)
        (st.success if passed == len(results) else st.error)(
            f"{passed} dari {len(results)} pengujian lulus."
        )
        st.table(results)


# ------------------------------------------------------------------- Tentang
def page_about():
    st.header("ℹ️ Tentang EmberNote")
    st.markdown(
        """
**EmberNote** adalah aplikasi pesan rahasia sekali baca (*burn-after-reading*).

**Algoritma tunggal: AES-256-GCM**
- Kunci 256-bit dibuat acak oleh CSPRNG sistem operasi (`os.urandom`), tidak diturunkan dari password.
- Nonce 96-bit baru dibuat untuk setiap enkripsi.
- **AAD** mengikat `ID pesan` dan `waktu kedaluwarsa` ke ciphertext, sehingga mengubah metadata membuat dekripsi gagal.
- Server hanya menyimpan nonce dan ciphertext+tag. Kunci tidak pernah disimpan.

**Alur**
1. Pengirim menulis pesan, aplikasi membuat kunci acak, mengenkripsi, dan menyimpan ciphertext.
2. Pengirim membagikan *ID* dan *kunci* lewat dua jalur berbeda.
3. Penerima memasukkan keduanya, pesan didekripsi dan diverifikasi, lalu dihapus dari server.

*Proyek pembelajaran kriptografi, bukan alat keamanan untuk produksi.*
        """
    )
    with st.expander("⚠️ Batasan & ancaman yang tidak ditangani"):
        st.markdown(
            """
- **Bukan E2EE murni**: kunci dibuat dan dipakai di server saat pesan dibuat/dibuka.
- Siapa pun yang memegang **ID + kunci** dapat membuka pesan, jadi kirim keduanya lewat jalur berbeda.
- **Panjang pesan, ID, dan waktu** tidak tersembunyi.
- Tidak ada pembatasan spam/percobaan (menebak kunci 256-bit tetap mustahil).
- Penyimpanan Streamlit Cloud bersifat sementara.
- Plaintext tetap ada di sesi browser sampai pesan ditutup.
- Belum diaudit pihak independen.
            """
        )


if "page" not in st.session_state:
    st.session_state.page = PAGES[1] if "id" in st.query_params else PAGES[0]

if LOGO.exists():
    st.sidebar.image(str(LOGO), width=96)
st.sidebar.title("EmberNote")
choice = st.sidebar.radio("Menu", PAGES, key="page")
st.sidebar.caption("AES-256-GCM • Pesan sekali baca")

tema.hero("EmberNote", "Pesan rahasia sekali baca, dilindungi satu algoritma.",
          ("AES-256-GCM", "Sekali baca", "AAD"))
{PAGES[0]: page_create, PAGES[1]: page_read, PAGES[2]: page_proof, PAGES[3]: page_about}[choice]()

In [ ]:
%%writefile requirements.txt
streamlit>=1.37
cryptography>=42.0

In [ ]:
%%writefile .gitignore
__pycache__/
*.pyc
*.db
.venv/
venv/
.ipynb_checkpoints/
.streamlit/secrets.toml
.DS_Store

## 6. Menjalankan dan Deploy
**Lokal:** buka terminal di folder proyek, lalu jalankan
```bash
pip install -r requirements.txt
streamlit run app.py
```
**Streamlit Community Cloud:**
1. Upload seluruh isi folder proyek ke repository GitHub.
2. Buka share.streamlit.io, login dengan GitHub, klik **Create app**.
3. Pilih repository dan branch, isi *Main file path* dengan `app.py`, lalu **Deploy**.

## 7. Checklist Screenshot untuk Laporan (format jpg)
1. Halaman **Buat Pesan** setelah pesan dibuat (ID, kunci, dan expander isi server).
2. Halaman **Baca Pesan** saat berhasil dibuka.
3. Halaman **Baca Pesan** saat kunci salah ditolak.
4. Percobaan kedua membaca pesan yang sama ("tidak ditemukan / sudah dibaca").
5. Halaman **Bukti Keamanan** setelah menekan *Jalankan semua pengujian* (9/9 lulus).

## 8. Batasan dan Ancaman yang Tidak Ditangani
Kelemahan berikut berada **di luar algoritma** dan tidak bisa ditambal dengan menambah cipher lain:
1. **Bukan E2EE murni.** Kunci dibuat dan dipakai di server saat pesan dibuat/dibuka; server hanya tidak *menyimpan* kunci dan plaintext.
2. **Siapa pun yang memegang ID + kunci dapat membuka pesan**, sehingga keduanya sebaiknya dikirim lewat jalur berbeda.
3. **Panjang pesan, ID, dan waktu dibuat/kedaluwarsa tidak tersembunyi.**
4. **Tidak ada pembatasan spam atau percobaan** (menebak kunci 256-bit tetap mustahil secara komputasi).
5. **Penyimpanan sementara** di Streamlit Cloud: pesan yang belum dibaca hilang saat aplikasi restart.
6. **Plaintext tetap ada di sesi browser** sampai pesan ditutup.
7. **Belum diaudit** pihak independen; ini proyek pembelajaran.

## 9. Kesimpulan
Untuk kerahasiaan, integritas data, dan keaslian metadata, **AES-256-GCM saja sudah cukup**: tidak ada kelemahan pada tingkat algoritma yang perlu ditambal algoritma lain, dan sembilan pengujian di atas menjadi buktinya. Batasan pada bagian 8 adalah batasan desain aplikasi, bukan celah algoritma.